Imports

In [44]:
import os
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix

Extract ZIP dataset

In [45]:
zip_path = "tdcsfog.zip"
extract_path = "tdcsfog_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

data_path = os.path.join(extract_path, "tdcsfog")

csv_files = [
    os.path.join(data_path, f)
    for f in os.listdir(data_path)
    if f.endswith(".csv")
]

print("Total files:", len(csv_files))

Total files: 833


Medication mapping

In [46]:
meta = pd.read_csv("tdcsfog_metadata.csv")
meta["med"] = meta["Medication"].map({"on": 1, "off": 0})
med_map = dict(zip(meta["Id"], meta["med"]))

Window settings

In [47]:
WINDOW_SIZE = 192
STEP = 128

Window extraction function

In [48]:
def extract_windows(df, med):

    df["FOG"] = (
        (df["StartHesitation"] == 1) |
        (df["Turn"] == 1) |
        (df["Walking"] == 1)
    ).astype(int)

    signals = df[["AccV", "AccML", "AccAP"]].values
    fog = df["FOG"].values

    X, y, m = [], [], []

    for start in range(0, len(df) - WINDOW_SIZE, STEP):
        end = start + WINDOW_SIZE

        ratio = fog[start:end].mean()

        if ratio >= 0.7:
            label = 1
        elif ratio <= 0.3:
            label = 0
        else:
            continue

        X.append(signals[start:end])
        y.append(label)
        m.append(med)

    return np.array(X), np.array(y), np.array(m)

Dataset construction

In [49]:
X_list, y_list, m_list, subj_list = [], [], [], []

for path in tqdm(csv_files):

    df = pd.read_csv(path)

    sid = os.path.basename(path).replace(".csv", "")
    med = med_map.get(sid, 0)

    X, y, m = extract_windows(df, med)

    if len(X) == 0:
        continue

    X_list.append(X)
    y_list.append(y)
    m_list.append(m)
    subj_list.extend([sid] * len(y))

X = np.concatenate(X_list)
y = np.concatenate(y_list)
m = np.concatenate(m_list)
subjects = np.array(subj_list)

print(X.shape, y.shape)

100%|██████████| 833/833 [00:08<00:00, 99.14it/s] 


(52984, 192, 3) (52984,)


Subject-wise split

In [50]:
unique_subj = np.unique(subjects)

train_subj, temp_subj = train_test_split(unique_subj, test_size=0.3, random_state=42)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.5, random_state=42)

def mask(subjs):
    return np.isin(subjects, subjs)

X_train, y_train, m_train = X[mask(train_subj)], y[mask(train_subj)], m[mask(train_subj)]
X_val, y_val, m_val = X[mask(val_subj)], y[mask(val_subj)], m[mask(val_subj)]
X_test, y_test, m_test = X[mask(test_subj)], y[mask(test_subj)], m[mask(test_subj)]

PyTorch Dataset

In [51]:
class FoGDataset(Dataset):
    def __init__(self, X, y, m):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1,1)
        self.m = torch.tensor(m, dtype=torch.float32).view(-1,1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]

        mean = x.mean(dim=0, keepdim=True)
        std = x.std(dim=0, keepdim=True) + 1e-8
        x = (x - mean) / std

        return x, self.y[idx], self.m[idx]

DataLoaders

In [52]:
train_loader = DataLoader(FoGDataset(X_train,y_train,m_train), batch_size=64, shuffle=True)
val_loader = DataLoader(FoGDataset(X_val,y_val,m_val), batch_size=64)
test_loader = DataLoader(FoGDataset(X_test,y_test,m_test), batch_size=64)

Model (CNN + LSTM + Medication conditioning)

In [53]:
class FoGModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(3, 32, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)

        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.3)

        self.med_embed = nn.Linear(1, 64)
        self.gamma = nn.Linear(64, 64)
        self.beta = nn.Linear(64, 64)

        self.lstm = nn.LSTM(
            64, 64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.fc = nn.Linear(128, 1)

    def forward(self, x, m):

        x = x.permute(0,2,1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        med = self.med_embed(m)
        gamma = self.gamma(med).unsqueeze(-1)
        beta = self.beta(med).unsqueeze(-1)

        x = gamma * x + beta

        x = self.drop(x)

        x = x.permute(0,2,1)
        x, _ = self.lstm(x)

        x = x.mean(dim=1)

        return self.fc(x)

Loss function (Stable BCE)

In [54]:
class StableLoss(nn.Module):
    def __init__(self, pos_weight=None):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        targets = targets.to(logits.device).float()

        pw = None
        if self.pos_weight is not None:
            pw = self.pos_weight.to(logits.device)

        return nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=pw
        )

Training loop

In [55]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FoGModel().to(device)

pos_weight = torch.tensor(
    [(y_train == 0).sum() / (y_train == 1).sum()],
    dtype=torch.float32
).to(device)

criterion = StableLoss(pos_weight)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

best_f1 = 0

for epoch in range(15):

    model.train()

    for Xb, yb, mb in train_loader:

        Xb = Xb.to(device)
        mb = mb.to(device)
        yb = yb.to(device).float()

        optimizer.zero_grad()

        out = model(Xb, mb)

        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

    # VALIDATION
    model.eval()

    probs, labels = [], []

    with torch.no_grad():
        for Xb, yb, mb in val_loader:

            Xb = Xb.to(device)
            mb = mb.to(device)

            p = torch.sigmoid(model(Xb, mb)).cpu().numpy()

            probs.extend(p)
            labels.extend(yb.numpy())

    probs = np.array(probs).flatten()
    labels = np.array(labels).flatten()

    preds = (probs > 0.5).astype(int)

    f1 = f1_score(labels, preds)

    print(f"Epoch {epoch+1} F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1 F1: 0.8489
Epoch 2 F1: 0.8765
Epoch 3 F1: 0.8823
Epoch 4 F1: 0.8796
Epoch 5 F1: 0.8743
Epoch 6 F1: 0.8883
Epoch 7 F1: 0.8865
Epoch 8 F1: 0.8820
Epoch 9 F1: 0.8838
Epoch 10 F1: 0.8862
Epoch 11 F1: 0.8899
Epoch 12 F1: 0.8771
Epoch 13 F1: 0.8843
Epoch 14 F1: 0.8863
Epoch 15 F1: 0.8777


Smoothing function

In [56]:
def smooth(preds, k=7):
    out = preds.copy()

    for i in range(len(preds)):
        start = max(0, i-k)
        if np.mean(preds[start:i+1]) < 0.5:
            out[i] = 0

    return out

Final evaluation

In [57]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

probs, labels = [], []

with torch.no_grad():
    for Xb, yb, mb in test_loader:

        Xb = Xb.to(device)
        mb = mb.to(device)

        p = torch.sigmoid(model(Xb, mb)).cpu().numpy()

        probs.extend(p)
        labels.extend(yb.numpy())

probs = np.array(probs).flatten()
labels = np.array(labels).flatten()

best_thresh = 0.42

preds = (probs > best_thresh).astype(int)
preds = smooth(preds)

print("F1:", f1_score(labels, preds))
print(classification_report(labels, preds))
print(confusion_matrix(labels, preds))

F1: 0.9276489028213166
              precision    recall  f1-score   support

         0.0       0.92      0.97      0.95      5217
         1.0       0.96      0.90      0.93      4111

    accuracy                           0.94      9328
   macro avg       0.94      0.93      0.94      9328
weighted avg       0.94      0.94      0.94      9328

[[5052  165]
 [ 412 3699]]
